# Patchwork Training — Closer to nnUNet

Same data as `patchwork_train.ipynb` (NAKO 100 subjects, 4-channel GRE + 82-class masks)  
but with parameters tuned to close the gap with `nnUNetResEncUNetMPlans_30G`.

## Changes vs baseline notebook

| Parameter | Baseline | This notebook | Rationale |
|---|---|---|---|
| `normalize_input` | `"mean"` | `None` | Replaced by per-channel CT normalization below |
| CT normalization | none | clip p0.5–p99.5 → z-score | Matches nnUNet's `CTNormalization`, uses nnUNet-computed stats |
| `destvox_rel` | `[3, 3, 3]` | `[3, 3, 2]` | Native z spacing is 3.0 mm (2× coarser than xy 1.41 mm); was applying same 3× downsampling in z as xy, giving 9 mm/vox in z vs 4.2 mm/vox in xy. Now 6 mm/vox in z → 2× native, closer to the 2:1 native ratio |
| `feature_dim` | `[8,16,16,32,64]` | `[16,32,32,64,128]` | 2× channels → 4× capacity per block (~6 M params total vs 1.5 M). VRAM-neutral because `batch_size` halved |
| `batch_size` | `4` | `2` | Halved to compensate for 2× feature channels (activation memory scales linearly with channels) |
| `num_patches` | `64` | `48` | Slight reduction to give VRAM headroom (~25% less patch storage) |
| `samples_per_it` | `2` | `2` | Kept at 2 — each label volume is ~8 GiB on GPU; 4 subjects caused fragmentation OOM |

## 0 — Paths & Imports

In [2]:
%load_ext autoreload
%autoreload 2

import os
# Use the async CUDA allocator — avoids fragmentation when large tensors
# (full-volume label arrays ~8 GiB each) compete with many small allocations.
# Must be set before TF initialises.
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

import sys, glob, gc
import numpy as np
import nibabel as nib
import tensorflow as tf
from tensorflow.keras import layers
from pathlib import Path
from hydra import compose, initialize
from omegaconf import OmegaConf

# Patchwork repo
sys.path.insert(0, "/nfs/norasys/notebooks/camaret/repos/patchwork")
import patchwork

# GPU memory growth
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

with initialize(config_path="dev/whole_body_benchmark/configs", version_base=None):
    cfg = compose(config_name="config")
print(OmegaConf.to_yaml(cfg))

I0000 00:00:1776329712.112743      80 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
W0000 00:00:1776329726.142655      80 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


paths:
  nako_dir: /nfs/data/nii/data0/GNC/GNC_759
  nnUNet_dir: /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/nnUNet/



## 1 — Find Valid Subjects

In [3]:
img_base  = Path(cfg.paths.nako_dir) / "links"
mask_base = Path(cfg.paths.nako_dir) / "data"
img_glob  = "30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.nii"
mask_rel  = "30/wholebody/total_vibe_and_ours_seg.nii"

train_dir = Path(cfg.paths.nnUNet_dir) / "nnUNet_raw/Dataset002_Nako_100_subjects/imagesTr"
val_dir   = Path(cfg.paths.nnUNet_dir) / "nnUNet_raw/Dataset002_Nako_100_subjects/imagesTs"

train_imgs   = train_dir.glob("*.nii.gz")
train_ids    = [f.stem.split("_")[0] for f in train_imgs]
subjects_tr  = list(set(train_ids))

val_imgs     = val_dir.glob("*.nii.gz")
val_ids      = [f.stem.split("_")[0] for f in val_imgs]
subjects_val = list(set(val_ids))

print(f"Train: {len(subjects_tr)}  Val: {len(subjects_val)}")

Train: 80  Val: 22


## 2 — CT Normalization + Build Data Dicts

**Change:** replaced `normalize_input: "mean"` (subtract global mean) with per-channel
CT normalization matching nnUNet's `CTNormalization`:
1. Clip to foreground percentiles [p0.5, p99.5] computed by nnUNet on this dataset.
2. Z-score normalise using foreground mean/std.

Stats source: `nnUNetResEncUNetMPlans_30G.json` → `foreground_intensity_properties_per_channel`.

In [4]:
# Per-channel intensity stats from nnUNet plans
# (foreground_intensity_properties_per_channel in nnUNetResEncUNetMPlans_30G.json)
CHANNEL_STATS = {
    0: {"mean": 200.775, "std": 147.837, "p005":  4.002, "p995": 678.997},
    1: {"mean": 251.370, "std": 156.140, "p005":  3.997, "p995": 733.009},
    2: {"mean": 162.197, "std": 181.455, "p005":  1.001, "p995": 698.010},
    3: {"mean":  96.874, "std":  78.965, "p005":  1.996, "p995": 309.999},
}

def ct_normalize(data: np.ndarray, ch: int) -> np.ndarray:
    """Clip to [p0.5, p99.5] then z-score. Matches nnUNet CTNormalization."""
    s = CHANNEL_STATS[ch]
    data = np.clip(data, s["p005"], s["p995"])
    data = (data - s["mean"]) / s["std"]
    return data


# Cache dir is separate from baseline to avoid mixing normalizations
CACHE_DIR = Path(cfg.paths.nako_dir) / "patchwork_cache_ctnorm"
CACHE_DIR.mkdir(exist_ok=True)

def get_channel_path(subject, ch, img_base, img_glob, cache_dir):
    dst = cache_dir / f"{subject}_ch{ch:04d}.nii.gz"
    if dst.exists():
        return str(dst)
    files = glob.glob(str(img_base / subject / img_glob))
    if not files:
        return None
    img  = nib.load(files[0])
    data = img.get_fdata(dtype=np.float32)
    ch_data = data[..., ch] if data.ndim == 4 else data
    # Apply CT normalization before caching
    ch_data = ct_normalize(ch_data, ch)
    out = nib.Nifti1Image(ch_data, img.affine, img.header)
    out.header.set_data_shape(ch_data.shape)
    nib.save(out, dst)
    return str(dst)


all_subjects_used = subjects_tr + subjects_val
contrasts    = [{} for _ in range(4)]
labels_dict  = {}

print("Preparing file references (CT-normalising and splitting channels on first run)...")
for subject in all_subjects_used:
    for ch in range(4):
        path = get_channel_path(subject, ch, img_base, img_glob, CACHE_DIR)
        if path:
            contrasts[ch][subject] = path
    mask_path = str(mask_base / subject / mask_rel)
    if os.path.exists(mask_path):
        labels_dict[subject] = mask_path

valid = [
    s for s in all_subjects_used
    if all(s in contrasts[ch] for ch in range(4)) and s in labels_dict
]
print(f"{len(valid)} subjects ready")

contrasts   = [{s: d[s] for s in valid if s in d} for d in contrasts]
labels_pw   = [{s: labels_dict[s] for s in valid}]
subjects_pw = valid
valid_ids   = list(range(len(subjects_tr), len(valid)))

Preparing file references (CT-normalising and splitting channels on first run)...
102 subjects ready


## 3 — Patchwork Config

### Key geometry change: `destvox_rel`

Native voxel spacing is **[3.0, 1.41, 1.41] mm** (z is 2× coarser than xy).  
The baseline used `destvox_rel=[3,3,3]`, giving level-2 voxels of **[4.2, 4.2, 9.0] mm** — 
3× native in every axis, including z which was already coarse.

With `destvox_rel=[3, 3, 2]`:
- xy: 3× native → **4.2 mm** (unchanged)
- z: 2× native → **6.0 mm** (was 9.0 mm)

The z/xy resolution ratio at level 2 goes from 9/4.2 = **2.1** down to 6/4.2 = **1.4**,
much closer to the native 3.0/1.41 = **2.1** and more importantly halves the z resolution gap.
nnUNet trains at native spacing so this brings us directionally closer.

### Capacity change: `feature_dim`

Doubled to `[16, 32, 32, 64, 128]` (was `[8, 16, 16, 32, 64]`).  
`batch_size` halved `4→2` to keep VRAM neutral (activation memory ∝ channels × batch).

In [6]:
nD = 3
MODEL_PATH = str(
    Path(cfg.paths.nako_dir).parent
    / "ANALYSIS_whole_body_benchmark/patchwork/models/whole_body_ctnorm_v2"
)
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

# --- Patching ---
# CHANGE: destvox_rel z-axis reduced 3→2 to respect native anisotropy [3.0, 1.41, 1.41] mm
patching = {
    "depth": 3,
    "ndim": nD,
    "scheme": {
        "patch_size": [32, 32, 32],
        "destvox_mm": None,
        "destvox_rel": [3, 3, 2],   # was [3,3,3] — z now 2× native instead of 3×
        "fov_mm": None,
        "fov_rel": [0.9, 0.9, 0.9],
    },
    "smoothfac_data": 0,
    "smoothfac_label": 0,
    "categorial_label": None,
    "interp_type": "NN",
    "scatter_type": "NN",
    "normalize_input": None,   # CHANGE: disabled — CT normalization applied at cache time
}

# --- Data loading (unchanged) ---
loading = {
    "nD": nD,
    "align_physical": False,
    "crop_fdim": None,
    "crop_fdim_labels": None,
    "crop_only_nonzero": False,
    "threshold": None,
    "add_inverted_label": False,
    "one_hot_index_list": list(range(1, 83)),
    "exclude_incomplete_labels": 1,
    "annotations_selector": None,
}

# --- Training ---
# CHANGE: num_patches 64→48 (frees ~25% patch VRAM), batch_size 4→2
training = {
    "num_patches": 48,       # was 64 — slight reduction for VRAM headroom
    "epochs": 10,
    "num_its": 5,
    "balance": {"ratio": 0.5, "autoweight": 1},
    "fit_type": "custom",
    "augment": {"dphi": 0.1, "flip": [1, 0, 0], "dscale": [0.1, 0.1, 0.1]},
}

outer_its      = 20
samples_per_it = 2    # Reverted 4→2: each label volume is ~8 GiB on GPU (320×260×316×82×float32).
                      # 4 subjects caused fragmentation OOM — BFC had 14 GiB free but no
                      # contiguous 8 GiB block. TF_GPU_ALLOCATOR=cuda_malloc_async (cell 0)
                      # also helps, but 2 subjects/iter is the safe limit on this GPU.

## 4 — Load One Sample & Build Model

In [7]:
def get_data(subjects_subset=None, n=None):
    subs = subjects_subset if subjects_subset else subjects_pw
    ctrs = [{s: d[s] for s in subs if s in d} for d in contrasts]
    lbls = [{s: labels_pw[0][s] for s in subs if s in labels_pw[0]}]
    return patchwork.improc_utils.load_data_structured(
        contrasts=ctrs, labels=lbls, subjects=subs, max_num_data=n, **loading
    )

print("Loading one subject to initialise model...")
with tf.device("/cpu:0"):
    tset, lset, rset, subjs = get_data(n=1)

print(f"Image shape: {tset[0].shape}   Label shape: {lset[0].shape}")
num_labels = lset[0].shape[nD + 1]
print(f"num_labels = {num_labels}")

Loading one subject to initialise model...
going to load 1 items
X
Image shape: (1, 320, 260, 316, 4)   Label shape: (1, 320, 260, 316, 82)
num_labels = 82


In [ ]:
reinit_model = False

if os.path.isfile(MODEL_PATH + ".json") and not reinit_model:
    print("Loading existing model...")
    themodel = patchwork.PatchWorkModel.load(MODEL_PATH, immediate_init=True, notmpfile=True)
else:
    print("Creating new model...")
    network = {
        "num_labels": num_labels,
        "modelname": MODEL_PATH,
        # CHANGE: feature_dim doubled [8,16,16,32,64] → [16,32,32,64,128]
        # batch_size halved 4→2 to keep VRAM neutral
        "blockCreator": lambda level, outK, input_shape:
            patchwork.customLayers.createUnet_v2(
                depth=5, outK=outK, nD=nD,
                input_shape=input_shape,
                feature_dim=[16, 32, 32, 64, 128],   # was [8, 16, 16, 32, 64]
            ),
        "intermediate_out": 8,
        "intermediate_loss": True,
        "finalBlock": layers.Activation("sigmoid"),
    }

    cgen     = patchwork.CropGenerator(**patching)
    themodel = patchwork.PatchWorkModel(cgen, **network)

    print("Initialising weights...")
    themodel.apply_full(
        tset[0][0:1, ...], resolution=rset[0],
        repetitions=10, generate_type="random", verbose=True,
    )

themodel.summary()

Creating new model...
Initialising weights...
>>> sampling patches for testing
input:  shape:[320,260,316]  width(mm):[448.6,364.2,945.0]  voxsize:[1.41,1.41,3.00]
snapper 0 1 1
level 0:  shape:[32,32,32]->[32,32,32]  width(mm):[403.7,327.8,850.5]
  voxsize:[13.02,10.57,27.44]  (rel. to input:[9.26,7.52,9.15]) 
  dest_shape:[35,35,35]
level 1:  shape:[32,32,32]->[32,32,32]  width(mm):[229.8,207.0,397.7]
  voxsize:[7.41,6.68,12.83]  (rel. to input:[5.27,4.75,4.28]) 
  dest_shape:[62,56,75]
level 2:  shape:[32,32,32]->[32,32,32]  width(mm):[130.8,130.8,186.0]
  voxsize:[4.22,4.22,6.00]  (rel. to input:[3.00,3.00,2.00]) 
  dest_shape:[107,87,158]


I0000 00:00:1776265597.767630  177628 cuda_solvers.cc:175] Creating GpuSolver handles for stream 0x61e4c4939b20


--------- cropping, level  0
 #patches:10 time/patch:111.358ms
 elapsed: 1113.580ms
--------- cropping, level  1
 #patches:10 time/patch:2.071ms
 elapsed: 20.713ms
--------- cropping, level  2
 #patches:10 time/patch:1.902ms
 elapsed: 19.020ms

>>> time elapsed, sampling: 2.935272827744484
>>> applying network --- 1/1


I0000 00:00:1776265599.825631  177628 cuda_dnn.cc:461] Loaded cuDNN version 90500


>>> time elapsed, network application: 2.205812368541956
>>> stitching result
>>> #patches last level: 10
>>> coverage: 8%
>>> time elapsed, stitching: 0.12589327991008759
>>> total time elapsed: 10.908351793885231


Model: "patch_work_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ activation (Activation)         │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cn_nblock (CNNblock)            │ ?                      │     1,799,562 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cn_nblock_1 (CNNblock)          │ ?                      │     1,877,322 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cn_nblock_2 (CNNblock)          │ ?                      │     1,870,402 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,547,286 (21.16 MB)

 Trainable params: 5,542,390 (21.14 MB)

 Non-trainable params: 4,896 (19.12 KB)

### Verify level geometry

Expected with `destvox_rel=[3,3,2]` on native spacing [1.41, 1.41, 3.0] mm:

| Level | xy voxel | z voxel | xy FoV | z FoV |
|---|---|---|---|---|
| 2 (fine) | ~4.2 mm | **~6.0 mm** (was 9.0) | ~135 mm | ~192 mm |
| 1 (med)  | ~7.4 mm | ~12 mm | ~237 mm | ~384 mm |
| 0 (coarse) | ~13 mm | ~21 mm | ~416 mm | ~672 mm |

## 5 — Training Loop

In [ ]:
import random

train_subjects = [s for s in subjects_pw if s not in [subjects_pw[i] for i in valid_ids]]

for i in range(outer_its):
    print(f"\n{'='*60}\nOuter iteration {i+1}/{outer_its}")

    subset = random.sample(train_subjects, min(samples_per_it, len(train_subjects)))

    with tf.device("/cpu:0"):
        tset, lset, rset, subjs = get_data(subjects_subset=subset)

    if len(tset) == 0:
        print("No data loaded, skipping")
        continue

    themodel.train(
        tset, lset, resolutions=rset,
        **training,
        batch_size=1,          # CHANGE: was 4 — halved to compensate for 2× feature channels
        valid_ids=[],
        autosave=True,
        verbose=2,
        inc_train_cycle=False,
        debug=False,
        patch_on_cpu=False,
    )

    with tf.device("/cpu:0"):
        del tset, lset
        gc.collect()

print("\nTraining complete.")

## 6 — Inference on a Validation Subject

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Label names from nnUNet dataset.json (index = class value in mask)
LABEL_NAMES = {
     1:"spleen", 2:"gallbladder", 3:"stomach", 4:"pancreas",
     5:"adrenal_gland_right", 6:"adrenal_gland_left",
     7:"lung_upper_lobe_left", 8:"lung_lower_lobe_left",
     9:"lung_upper_lobe_right", 10:"lung_middle_lobe_right", 11:"lung_lower_lobe_right",
    12:"esophagus", 13:"trachea", 14:"thyroid_gland", 15:"intestine", 16:"duodenum",
    17:"unused", 18:"urinary_bladder", 19:"sacrum", 20:"heart",
    21:"pulmonary_vein", 22:"brachiocephalic_trunk",
    23:"subclavian_artery_right", 24:"subclavian_artery_left",
    25:"common_carotid_artery_right", 26:"common_carotid_artery_left",
    27:"brachiocephalic_vein_left", 28:"brachiocephalic_vein_right",
    29:"atrial_appendage_left", 30:"superior_vena_cava", 31:"inferior_vena_cava",
    32:"portal_vein_and_splenic_vein",
    33:"iliac_artery_left", 34:"iliac_artery_right",
    35:"iliac_vena_left", 36:"iliac_vena_right",
    37:"humerus_left", 38:"humerus_right",
    39:"scapula_left", 40:"scapula_right",
    41:"clavicula_left", 42:"clavicula_right",
    43:"femur_left", 44:"femur_right",
    45:"hip_left", 46:"hip_right",
    47:"spinal_cord",
    48:"gluteus_maximus_left", 49:"gluteus_maximus_right",
    50:"gluteus_medius_left", 51:"gluteus_medius_right",
    52:"gluteus_minimus_left", 53:"gluteus_minimus_right",
    54:"autochthon_left", 55:"autochthon_right",
    56:"iliopsoas_left", 57:"iliopsoas_right",
    58:"sternum", 59:"costal_cartilages", 60:"IVD",
    61:"vertebra_body", 62:"vertebra_posterior_elements", 63:"spinal_channel",
    64:"bone_other",
    65:"AT_pelvis", 66:"IMAT_pelvis", 67:"Muscle_pelvis", 68:"AVAT_pelvis",
    69:"AT_upper_abdomen", 70:"IMAT_upper_abdomen", 71:"Muscle_upper_abdomen", 72:"AVAT_upper_abdomen",
    73:"AT_thorax", 74:"IMAT_thorax", 75:"Muscle_thorax", 76:"AVAT_thorax",
    77:"liver", 78:"pankreas", 79:"prostate", 80:"aorta",
    81:"kidney_cortex", 82:"kidney_hilus",
}

# ── Inference ────────────────────────────────────────────────────────────────
val_subject = subjects_pw[valid_ids[0]]
print(f"Running inference on: {val_subject}")

with tf.device("/cpu:0"):
    tset_val, lset_val, rset_val, _ = get_data(subjects_subset=[val_subject])

pred = themodel.apply_full(
    tset_val[0][0:1, ...],
    resolution=rset_val[0],
    num_patches=200,
    generate_type="random",
    verbose=2,
)
# apply_full returns [H, W, D, num_labels] — batch dim squeezed internally
pred = np.array(pred)
img  = np.array(tset_val[0])   # EagerTensor → numpy
print(f"pred shape: {pred.shape}")

pred_label = np.argmax(pred, axis=-1) + 1   # [H, W, D], 1-based
pred_label[np.max(pred, axis=-1) < 0.3] = 0

# Ground-truth: one-hot [H,W,D,82] → integer mask
gt_onehot = np.array(lset_val[0][0])        # [H, W, D, 82]
gt_label  = np.argmax(gt_onehot, axis=-1) + 1
gt_label[np.max(gt_onehot, axis=-1) == 0] = 0

# ── Dice per label ────────────────────────────────────────────────────────────
dices = np.full(num_labels, np.nan)
for i in range(num_labels):
    cls = i + 1
    p = (pred_label == cls)
    g = (gt_label   == cls)
    union = p.sum() + g.sum()
    if union > 0:
        dices[i] = 2.0 * (p & g).sum() / union

mean_dice = float(np.nanmean(dices))
print(f"\nMean Dice (NaN-safe, {np.sum(~np.isnan(dices))}/{num_labels} present classes): {mean_dice:.4f}")

# ── Slice visualisation ───────────────────────────────────────────────────────
mid_slice = pred_label.shape[2] // 2
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img[0, :, :, mid_slice, 0].T, cmap="gray", origin="lower")
axes[0].set_title("Input (channel 0)")
axes[1].imshow(gt_label[:, :, mid_slice].T, origin="lower", interpolation="nearest")
axes[1].set_title("Ground truth")
axes[2].imshow(pred_label[:, :, mid_slice].T, origin="lower", interpolation="nearest")
axes[2].set_title("Prediction")
for ax in axes:
    ax.axis("off")
plt.suptitle(f"Subject: {val_subject}  |  mid axial slice z={mid_slice}", y=1.01)
plt.tight_layout()
plt.show()

# ── Per-label Dice bar chart ──────────────────────────────────────────────────
valid_mask  = ~np.isnan(dices)
valid_idx   = np.where(valid_mask)[0]                       # indices into dices[]
valid_dices = dices[valid_mask]
valid_names = [f"{i+1} {LABEL_NAMES.get(i+1, f'cls_{i+1}')}" for i in valid_idx]

# Sort by Dice ascending for easier reading
order       = np.argsort(valid_dices)
sorted_d    = valid_dices[order]
sorted_n    = [valid_names[k] for k in order]
colors      = ["#d62728" if d < 0.5 else "#ff7f0e" if d < 0.7 else "#2ca02c" for d in sorted_d]

fig, ax = plt.subplots(figsize=(8, max(6, len(sorted_d) * 0.22)))
bars = ax.barh(range(len(sorted_d)), sorted_d, color=colors, height=0.8)
ax.set_yticks(range(len(sorted_d)))
ax.set_yticklabels(sorted_n, fontsize=7)
ax.set_xlabel("Dice coefficient")
ax.set_xlim(0, 1)
ax.axvline(mean_dice, color="black", linestyle="--", linewidth=1.2, label=f"mean = {mean_dice:.3f}")
ax.axvline(0.5, color="#d62728", linestyle=":", linewidth=0.8)
ax.axvline(0.7, color="#ff7f0e", linestyle=":", linewidth=0.8)
ax.legend(loc="lower right", fontsize=8)
ax.set_title(f"Per-label Dice — {val_subject}  (absent classes excluded)")
plt.tight_layout()
plt.show()

## 6b — Per-Label Dice Validation\n\nRuns inference on every validation subject, computes Dice per label (classes 1–82),\nand writes a summary text file that mirrors nnUNet's validation output format:\n- one row per subject in a per-case block\n- one row per label with mean ± std across subjects\n- overall mean Dice at the bottom

In [1]:
import datetime, json
# Label names from nnUNet dataset.json
# Keys are 1-based class indices matching 
# one_hot_index_list = range(1, 83)
LABEL_NAMES = {
     1:"spleen", 2:"gallbladder", 3:"stomach", 4:"pancreas",
     5:"adrenal_gland_right", 6:"adrenal_gland_left",
     7:"lung_upper_lobe_left", 8:"lung_lower_lobe_left",
     9:"lung_upper_lobe_right", 10:"lung_middle_lobe_right", 11:"lung_lower_lobe_right",
    12:"esophagus", 13:"trachea", 14:"thyroid_gland", 15:"intestine", 16:"duodenum",
    17:"unused", 18:"urinary_bladder", 19:"sacrum", 20:"heart",
    21:"pulmonary_vein", 22:"brachiocephalic_trunk",
    23:"subclavian_artery_right", 24:"subclavian_artery_left",
    25:"common_carotid_artery_right", 26:"common_carotid_artery_left",
    27:"brachiocephalic_vein_left", 28:"brachiocephalic_vein_right",
    29:"atrial_appendage_left", 30:"superior_vena_cava", 31:"inferior_vena_cava",
    32:"portal_vein_and_splenic_vein",
    33:"iliac_artery_left", 34:"iliac_artery_right",
    35:"iliac_vena_left", 36:"iliac_vena_right",
    37:"humerus_left", 38:"humerus_right",
    39:"scapula_left", 40:"scapula_right",
    41:"clavicula_left", 42:"clavicula_right",
    43:"femur_left", 44:"femur_right",
    45:"hip_left", 46:"hip_right",
    47:"spinal_cord",
    48:"gluteus_maximus_left", 49:"gluteus_maximus_right",
    50:"gluteus_medius_left", 51:"gluteus_medius_right",
    52:"gluteus_minimus_left", 53:"gluteus_minimus_right",
    54:"autochthon_left", 55:"autochthon_right",
    56:"iliopsoas_left", 57:"iliopsoas_right",
    58:"sternum", 59:"costal_cartilages", 60:"IVD",
    61:"vertebra_body", 62:"vertebra_posterior_elements", 63:"spinal_channel",
    64:"bone_other",
    65:"AT_pelvis", 66:"IMAT_pelvis", 67:"Muscle_pelvis", 68:"AVAT_pelvis",
    69:"AT_upper_abdomen", 70:"IMAT_upper_abdomen", 71:"Muscle_upper_abdomen", 72:"AVAT_upper_abdomen",
    73:"AT_thorax", 74:"IMAT_thorax", 75:"Muscle_thorax", 76:"AVAT_thorax",
    77:"liver", 78:"pankreas", 79:"prostate", 80:"aorta",
    81:"kidney_cortex", 82:"kidney_hilus",
}

def dice_per_label(
        pred_prob: np.ndarray,
        gt_mask: np.ndarray,
        num_labels: int = 82,
        bg_threshold: float = 0.3,
) -> np.ndarray:
    """
    Compute Dice coefficient for each label (1-based, 1..num_labels).

    Parameters
    ----------
    pred_prob : float32 array [H, W, D, num_labels]
        Sigmoid probabilities output by apply_full, one channel per class (0-indexed → class 1-based).
    gt_mask   : int array [H, W, D]
        Ground-truth integer label map (values 0..num_labels).
    bg_threshold : float
        Voxels where max pred < threshold are set to background.

    Returns
    -------
    dices : float32 array [num_labels]
        dices[i] = Dice for class (i+1). NaN when class absent in both pred and gt.
    """
    pred_label = np.argmax(pred_prob, axis=-1) + 1          # [H,W,D], values 1..num_labels
    pred_label[np.max(pred_prob, axis=-1) < bg_threshold] = 0

    dices = np.full(num_labels, np.nan, dtype=np.float32)
    for i in range(num_labels):
        cls = i + 1
        p = (pred_label == cls)
        g = (gt_mask   == cls)
        union = p.sum() + g.sum()
        if union == 0:
            continue   # class absent in both — leave as NaN
        dices[i] = 2.0 * (p & g).sum() / union
    return dices


print(f"Label names loaded for {len(LABEL_NAMES)} classes.")

NameError: name 'np' is not defined

In [ ]:
VAL_DICE_PATH = MODEL_PATH + "_val_dice.txt"
VAL_DICE_JSON = MODEL_PATH + "_val_dice.json"
VAL_NUM_PATCHES   = 300   # patches per subject for inference (more = better coverage)
VAL_BG_THRESHOLD  = 0.3   # voxels with max sigmoid < this → background
val_subjects_list = [subjects_pw[i] for i in valid_ids]
print(f"Evaluating {len(val_subjects_list)} validation subjects → {VAL_DICE_PATH}")
# per_subject_dices[subject] = float32 array [82]
per_subject_dices = {}
for subj_idx, subj in enumerate(val_subjects_list):
    print(f"  [{subj_idx+1}/{len(val_subjects_list)}] {subj}", end="", flush=True)

    # Load image + ground-truth
    with tf.device("/cpu:0"):
        tset_v, lset_v, rset_v, _ = get_data(subjects_subset=[subj])

    if len(tset_v) == 0:
        print(" — skipped (no data)")
        continue

    # Inference (full-volume patch-based)
    pred_prob = themodel.apply_full(
        tset_v[0][0:1, ...],
        resolution=rset_v[0],
        num_patches=VAL_NUM_PATCHES,
        generate_type="random",
        verbose=0,
    )
    # pred_prob: list → take [0], shape [H,W,D,82]
    pred_prob_vol = pred_prob[0] if isinstance(pred_prob, list) else pred_prob
    pred_prob_vol = np.array(pred_prob_vol, dtype=np.float32)

    # Ground-truth: one_hot → argmax back to integer mask
    gt_onehot = lset_v[0][0]          # [H,W,D,82]  (one-hot, classes 1..82)
    gt_int = np.argmax(gt_onehot, axis=-1) + 1          # 1..82
    gt_int[np.max(gt_onehot, axis=-1) == 0] = 0         # background voxels

    per_subject_dices[subj] = dice_per_label(
        pred_prob_vol, gt_int,
        num_labels=num_labels,
        bg_threshold=VAL_BG_THRESHOLD,
    )

    subj_mean = float(np.nanmean(per_subject_dices[subj]))
    print(f"  mean Dice = {subj_mean:.4f}")

    with tf.device("/cpu:0"):
        del tset_v, lset_v, pred_prob_vol, gt_onehot
        gc.collect()

print(f"\nDone. {len(per_subject_dices)} subjects evaluated.")

# ── Aggregate across subjects ────────────────────────────────────────────────
all_dices = np.stack(list(per_subject_dices.values()), axis=0)  # [N, 82]
label_mean = np.nanmean(all_dices, axis=0)   # [82]
label_std  = np.nanstd( all_dices, axis=0)   # [82]
overall_mean = float(np.nanmean(label_mean))

# ── Write text file (nnUNet-style) ───────────────────────────────────────────
lines = []
lines.append("=" * 72)
lines.append("Patchwork Validation — Dice per label")
lines.append(f"Model     : {MODEL_PATH}")
lines.append(f"Date      : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
lines.append(f"Val set   : {len(per_subject_dices)} subjects")
lines.append(f"Patches   : {VAL_NUM_PATCHES} per subject, bg_threshold={VAL_BG_THRESHOLD}")
lines.append("=" * 72)
lines.append("")

# Per-subject block
lines.append("── Per-subject mean Dice " + "-" * 48)
for subj, dices in per_subject_dices.items():
    lines.append(f"  {subj:<20s}  mean = {np.nanmean(dices):.4f}")
lines.append("")

# Per-label block
lines.append("── Per-label summary (mean ± std across subjects) " + "-" * 22)
col_w = max(len(n) for n in LABEL_NAMES.values()) + 2
for i in range(num_labels):
    cls  = i + 1
    name = LABEL_NAMES.get(cls, f"class_{cls}")
    m, s = label_mean[i], label_std[i]
    if np.isnan(m):
        lines.append(f"  {cls:>3d}  {name:<{col_w}s}  n/a (absent in all subjects)")
    else:
        lines.append(f"  {cls:>3d}  {name:<{col_w}s}  {m:.4f} ± {s:.4f}")
lines.append("")
lines.append(f"Overall mean Dice (macro, NaN-safe): {overall_mean:.4f}")
lines.append("=" * 72)

with open(VAL_DICE_PATH, "w") as f:
    f.write("\n".join(lines) + "\n")
print("\n".join(lines[-10:]))   # preview tail
print(f"\nFull results written to:\n  {VAL_DICE_PATH}")

# ── Write companion JSON (machine-readable) ──────────────────────────────────
results_json = {
    "model": MODEL_PATH,
    "date": datetime.datetime.now().isoformat(),
    "num_val_subjects": len(per_subject_dices),
    "bg_threshold": VAL_BG_THRESHOLD,
    "overall_mean_dice": overall_mean,
    "per_label": {
        LABEL_NAMES.get(i+1, f"class_{i+1}"): {
            "class_index": i + 1,
            "mean": float(label_mean[i]) if not np.isnan(label_mean[i]) else None,
            "std":  float(label_std[i])  if not np.isnan(label_std[i])  else None,
        }
        for i in range(num_labels)
    },
    "per_subject": {
        subj: {
            LABEL_NAMES.get(i+1, f"class_{i+1}"): (
                float(dices[i]) if not np.isnan(dices[i]) else None
            )
            for i in range(num_labels)
        }
        for subj, dices in per_subject_dices.items()
    },
}
with open(VAL_DICE_JSON, "w") as f:
    json.dump(results_json, f, indent=2)
print(f"JSON written to:\n  {VAL_DICE_JSON}")   

In [ ]:
import matplotlib.pyplot as plt

val_subject = subjects_pw[valid_ids[0]]
print(f"Running inference on: {val_subject}")

with tf.device("/cpu:0"):
    tset_val, lset_val, rset_val, _ = get_data(subjects_subset=[val_subject])

pred = themodel.apply_full(
    tset_val[0][0:1, ...],
    resolution=rset_val[0],
    num_patches=200,
    generate_type="random",
    verbose=2,
)
# apply_full returns [H, W, D, num_labels] — batch dim already squeezed by tf.squeeze
pred = np.array(pred)
img  = np.array(tset_val[0])   # EagerTensor → numpy before slicing/.T
print(f"pred shape: {pred.shape}")   # expect (H, W, D, 82)
pred_label = np.argmax(pred, axis=-1) + 1
pred_label[np.max(pred, axis=-1) < 0.3] = 0

mid_slice = pred_label.shape[2] // 2
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img[0, :, :, mid_slice, 0].T, cmap="gray", origin="lower")
axes[0].set_title("Input (channel 0, CT-normalised)")
axes[1].imshow(pred_label[:, :, mid_slice].T, origin="lower")
axes[1].set_title("Predicted segmentation")
plt.tight_layout()
plt.show()

## 7 — Save / Reload

In [ ]:
# Save is automatic when autosave=True; reload:
# themodel = patchwork.PatchWorkModel.load(MODEL_PATH, immediate_init=True)
print(f"Model saved at: {MODEL_PATH}")